# 🚗 Lane Detection — MobileNetV2-UNet for Jetson Nano
**Datasets:** Road Lane Segmentation (Roboflow YOLO format) + BDD100K  
**Architecture:** MobileNetV2 Encoder → ASPP Bridge → UNet Decoder → Binary Mask  
**Target:** ~3.5M params | ~35 FPS with TensorRT FP16 on Jetson Nano 4GB

## 📦 1. Install Dependencies

In [1]:
pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.5 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install -q albumentations onnx onnxruntime tensorboard
# torch & torchvision already available on Kaggle GPU kernels

## ⚙️ 2. Configuration

In [3]:
import os
import torch

class Config:
    # ─── Dataset Paths ───────────────────────────────────────────────────────────
    # Road Lane Segmentation dataset (Roboflow YOLO segmentation format)
    ROAD_LANE_DIR   = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels/dataset"
    # BDD100K segmentation masks
    BDD100K_SEG_DIR = "/kaggle/input/datasets/solesensei/solesensei_bdd100k/"

    # ─── Image Settings ──────────────────────────────────────────────────────────
    IMG_WIDTH  = 512   # optimal for Jetson Nano: good resolution, fits in 4GB
    IMG_HEIGHT = 256

    # ─── Training ────────────────────────────────────────────────────────────────
    BATCH_SIZE      = 16         # reduce to 8 if OOM
    NUM_EPOCHS      = 60
    LEARNING_RATE   = 1e-3
    WEIGHT_DECAY    = 1e-4
    NUM_WORKERS     = 4
    PIN_MEMORY      = True
    MIXED_PRECISION = True       # AMP — speeds training, reduces VRAM

    # ─── Model ───────────────────────────────────────────────────────────────────
    NUM_CLASSES = 1              # Binary: lane pixel vs background
    PRETRAINED  = True
    DROPOUT     = 0.2

    # ─── Loss weights ────────────────────────────────────────────────────────────
    BCE_WEIGHT  = 0.5
    DICE_WEIGHT = 0.5

    # ─── BDD100K lane class IDs (pixel values in seg PNG masks) ──────────────────
    BDD_LANE_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    # ─── Augmentation ────────────────────────────────────────────────────────────
    AUG_HFLIP_P        = 0.5
    AUG_BRIGHTNESS     = 0.3
    AUG_CONTRAST       = 0.3
    AUG_SATURATION     = 0.2
    AUG_HUE            = 0.1
    AUG_MOTION_BLUR_P  = 0.2
    AUG_COARSE_DROPOUT = 0.1

    # ─── Paths ───────────────────────────────────────────────────────────────────
    CHECKPOINT_DIR  = "/kaggle/working/checkpoints"
    BEST_MODEL_PATH = "/kaggle/working/checkpoints/best_model.pth"
    ONNX_PATH       = "/kaggle/working/lane_detection.onnx"
    TRT_ENGINE_PATH = "/kaggle/working/lane_detection_fp16.trt"
    LOG_DIR         = "/kaggle/working/logs"

    # ─── Inference ───────────────────────────────────────────────────────────────
    CONF_THRESHOLD = 0.5

    # ─── Device ──────────────────────────────────────────────────────────────────
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    @classmethod
    def make_dirs(cls):
        os.makedirs(cls.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.LOG_DIR, exist_ok=True)

cfg = Config()
print(f"Device : {cfg.DEVICE}")
print(f"AMP    : {cfg.MIXED_PRECISION}")
print(f"Image  : {cfg.IMG_WIDTH}×{cfg.IMG_HEIGHT}")

Device : cuda
AMP    : True
Image  : 512×256


## 🏗️ 3. Model — MobileNetV2-UNet + ASPP

In [4]:
"""
DeepLabV3+ with ASPP — fixed for batch_size=1 and eval/sanity-check usage.

Fixes applied
─────────────
1. gap branch uses GroupNorm instead of BatchNorm2d (BN can't handle 1×1 spatial maps).
2. build_and_check() calls model.eval() before the single-image sanity-check.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights


# ──────────────────────────────────────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────────────────────────────────────

class Config:
    IMG_HEIGHT   = 512
    IMG_WIDTH    = 512
    NUM_CLASSES  = 21          # VOC; change freely
    ASPP_OUT_CH  = 256
    DECODER_CH   = 48
    DROPOUT      = 0.5

cfg = Config()


# ──────────────────────────────────────────────────────────────────────────────
# Building blocks
# ──────────────────────────────────────────────────────────────────────────────

class ConvBnRelu(nn.Module):
    """Conv2d → BatchNorm2d → ReLU6.
    Safe for full spatial maps (H/4, H/8, H/16, H/32).

    padding is passed directly to Conv2d — do NOT multiply by dilation here.
    For a 3×3 atrous conv with dilation=d, pass padding=d (same-padding formula:
    padding = dilation * (kernel - 1) // 2 = d*1 = d).
    Multiplying again (padding * dilation) would over-pad and produce a larger
    output than the input, causing the cat() size mismatch in ASPP.
    """
    def __init__(self, in_ch, out_ch, kernel=3, stride=1,
                 padding=1, dilation=1, bias=False):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride,
                      padding=padding, dilation=dilation, bias=bias),  # ← fixed
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# ──────────────────────────────────────────────────────────────────────────────
# ASPP module  ← main fix lives here
# ──────────────────────────────────────────────────────────────────────────────

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling.

    The global-average-pool (gap) branch collapses spatial dims to 1×1 before
    normalising.  BatchNorm2d in training mode requires >1 spatial value, so it
    crashes with any batch size when the feature map becomes 1×1.

    Fix: replace BatchNorm2d in the gap branch with GroupNorm(num_groups, ch).
    GroupNorm normalises over channel groups independently of spatial size, so
    it works correctly on 1×1 tensors.
    """

    def __init__(self, in_ch: int, out_ch: int = 256,
                 dilations: tuple = (6, 12, 18)):
        super().__init__()

        # 1×1 convolution
        self.conv1 = ConvBnRelu(in_ch, out_ch, kernel=1, padding=0)

        # Atrous convolutions at three scales
        self.atrous = nn.ModuleList([
            ConvBnRelu(in_ch, out_ch, dilation=d, padding=d) for d in dilations
        ])

        # ── Global Average Pool branch ──────────────────────────────────────
        # BN after AdaptiveAvgPool2d(1) → always 1×1 → BN training crash.
        # GroupNorm does NOT depend on spatial resolution, so it is safe here.
        num_groups = 32  # must divide out_ch evenly (256 / 32 = 8 ✓)
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(num_groups, out_ch),   # ← was nn.BatchNorm2d(out_ch)
            nn.ReLU6(inplace=True),
        )
        # ────────────────────────────────────────────────────────────────────

        # Project concatenated branches back to out_ch
        total_branches = 1 + len(dilations) + 1   # conv1 + atrous + gap
        self.project = nn.Sequential(
            ConvBnRelu(out_ch * total_branches, out_ch, kernel=1, padding=0),
            nn.Dropout(0.5),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h, w = x.shape[-2:]

        gap = F.interpolate(self.gap(x), size=(h, w),
                            mode="bilinear", align_corners=False)
        feats = [self.conv1(x)] + [a(x) for a in self.atrous] + [gap]
        return self.project(torch.cat(feats, dim=1))


# ──────────────────────────────────────────────────────────────────────────────
# Encoder (ResNet-50 backbone)
# ──────────────────────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    """ResNet-50 with dilated conv in layer3/layer4 (output_stride=16)."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = resnet50(weights=weights)

        # Stem + layer1 → low-level features at stride 4
        self.layer0 = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1   # stride 4,  256 ch  (low-level)
        self.layer2 = backbone.layer2   # stride 8,  512 ch
        self.layer3 = backbone.layer3   # stride 16, 1024 ch  (dilated)
        self.layer4 = backbone.layer4   # stride 16, 2048 ch  (dilated)

        # Apply dilation to keep output_stride=16
        self._apply_dilation(self.layer3, dilation=2, stride=1)
        self._apply_dilation(self.layer4, dilation=4, stride=1)

    @staticmethod
    def _apply_dilation(layer, dilation, stride):
        for m in layer.modules():
            if isinstance(m, nn.Conv2d):
                if m.kernel_size == (3, 3):
                    m.dilation = (dilation, dilation)
                    m.padding  = (dilation, dilation)
                    m.stride   = (stride, stride)
            elif isinstance(m, nn.BatchNorm2d):
                pass
        # Fix the first downsample (stride conv) so spatial size stays at /16
        first_block = layer[0]
        if first_block.downsample is not None:
            first_block.downsample[0].stride = (stride, stride)
        first_block.conv2.stride = (stride, stride)

    def forward(self, x):
        x = self.layer0(x)
        low = self.layer1(x)    # /4  — returned for decoder skip connection
        x   = self.layer2(low)
        x   = self.layer3(x)
        x   = self.layer4(x)   # /16 — goes into ASPP
        return x, low


# ──────────────────────────────────────────────────────────────────────────────
# Decoder
# ──────────────────────────────────────────────────────────────────────────────

class Decoder(nn.Module):
    def __init__(self, low_ch: int = 256, aspp_ch: int = 256,
                 num_classes: int = 21, decoder_ch: int = 48):
        super().__init__()
        self.low_proj = ConvBnRelu(low_ch, decoder_ch, kernel=1, padding=0)

        self.refine = nn.Sequential(
            ConvBnRelu(aspp_ch + decoder_ch, 256),
            ConvBnRelu(256, 256),
        )
        self.cls = nn.Conv2d(256, num_classes, 1)

    def forward(self, aspp_feat, low_feat):
        low = self.low_proj(low_feat)                           # /4
        h, w = low.shape[-2:]
        aspp_up = F.interpolate(aspp_feat, size=(h, w),
                                mode="bilinear", align_corners=False)
        x = torch.cat([aspp_up, low], dim=1)
        x = self.refine(x)
        return self.cls(x)                                      # /4 logits


# ──────────────────────────────────────────────────────────────────────────────
# Full model
# ──────────────────────────────────────────────────────────────────────────────

class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes: int = cfg.NUM_CLASSES,
                 pretrained: bool = True):
        super().__init__()
        self.encoder = Encoder(pretrained=pretrained)
        self.aspp    = ASPP(in_ch=2048, out_ch=cfg.ASPP_OUT_CH)
        self.decoder = Decoder(
            low_ch=256,
            aspp_ch=cfg.ASPP_OUT_CH,
            num_classes=num_classes,
            decoder_ch=cfg.DECODER_CH,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h, w = x.shape[-2:]
        enc, low = self.encoder(x)
        aspp     = self.aspp(enc)
        logits   = self.decoder(aspp, low)
        # Upsample back to input resolution
        return F.interpolate(logits, size=(h, w),
                             mode="bilinear", align_corners=False)


# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def build_model(num_classes: int = cfg.NUM_CLASSES,
                pretrained: bool = True) -> DeepLabV3Plus:
    return DeepLabV3Plus(num_classes=num_classes, pretrained=pretrained)


def build_and_check():
    """Sanity-check: single-image forward pass (eval mode)."""
    model = build_model(pretrained=False)

    # ── Fix 2: always call eval() before a single-image sanity check ────────
    model.eval()
    # ────────────────────────────────────────────────────────────────────────

    x = torch.randn(1, 3, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)
    with torch.no_grad():
        y = model(x)

    print(f"Input  shape : {tuple(x.shape)}")
    print(f"Output shape : {tuple(y.shape)}")
    assert y.shape == (1, cfg.NUM_CLASSES, cfg.IMG_HEIGHT, cfg.IMG_WIDTH), \
        f"Unexpected output shape: {y.shape}"
    print("✓ Sanity check passed.")
    return model


# ──────────────────────────────────────────────────────────────────────────────
# Training utilities
# ──────────────────────────────────────────────────────────────────────────────

def get_optimizer(model: nn.Module, lr: float = 1e-4, weight_decay: float = 1e-4):
    """Separate LRs: 10× lower for the pretrained backbone."""
    backbone_ids = {id(p) for p in model.encoder.parameters()}
    backbone_params = [p for p in model.parameters() if id(p) in backbone_ids]
    head_params     = [p for p in model.parameters() if id(p) not in backbone_ids]
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": lr * 0.1},
        {"params": head_params,     "lr": lr},
    ], weight_decay=weight_decay)


def get_criterion(ignore_index: int = 255):
    return nn.CrossEntropyLoss(ignore_index=ignore_index)


# ──────────────────────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    build_and_check()

Input  shape : (1, 3, 512, 512)
Output shape : (1, 21, 512, 512)
✓ Sanity check passed.


## 📉 4. Losses & Metrics

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceLoss(nn.Module):
    """
    Dice Loss — critical for lane detection because lanes occupy
    very few pixels (heavy class imbalance).
    """
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        flat_p = probs.view(-1)
        flat_t = targets.view(-1)
        inter  = (flat_p * flat_t).sum()
        dice   = (2.0 * inter + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)
        return 1.0 - dice


class FocalLoss(nn.Module):
    """Focal Loss — down-weights easy background pixels."""
    def __init__(self, alpha: float = 0.8, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


class CombinedLoss(nn.Module):
    """
    BCE + Dice — best combination for lane segmentation:
      BCE : standard pixel-wise classification
      Dice: handles class imbalance from thin lane pixels
    pos_weight > 1 further penalizes missing lanes.
    """
    def __init__(self, bce_w: float = 0.5, dice_w: float = 0.5,
                 pos_weight: float = 5.0):
        super().__init__()
        self.bce_w  = bce_w
        self.dice_w = dice_w
        pw = torch.tensor([pos_weight])
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=pw)
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        self.bce.pos_weight = self.bce.pos_weight.to(logits.device)
        return self.bce_w * self.bce(logits, targets) + \
               self.dice_w * self.dice(logits, targets)


class LaneMetrics:
    """Accumulates batch metrics and returns epoch averages."""

    def __init__(self, threshold: float = 0.5, eps: float = 1e-7):
        self.threshold = threshold
        self.eps       = eps
        self.reset()

    def reset(self):
        self.total_iou  = 0.0
        self.total_dice = 0.0
        self.total_f1   = 0.0
        self.total_acc  = 0.0
        self.n_batches  = 0

    @torch.no_grad()
    def update(self, logits, targets):
        preds   = (torch.sigmoid(logits) > self.threshold).float()
        targets = targets.float()
        tp = (preds * targets).sum(dim=(2, 3))
        fp = (preds * (1 - targets)).sum(dim=(2, 3))
        fn = ((1 - preds) * targets).sum(dim=(2, 3))
        tn = ((1 - preds) * (1 - targets)).sum(dim=(2, 3))
        iou  = (tp / (tp + fp + fn + self.eps)).mean().item()
        dice = (2 * tp / (2 * tp + fp + fn + self.eps)).mean().item()
        prec = (tp / (tp + fp + self.eps)).mean().item()
        rec  = (tp / (tp + fn + self.eps)).mean().item()
        f1   = 2 * prec * rec / (prec + rec + self.eps)
        acc  = ((tp + tn) / (tp + fp + fn + tn + self.eps)).mean().item()
        self.total_iou  += iou
        self.total_dice += dice
        self.total_f1   += f1
        self.total_acc  += acc
        self.n_batches  += 1

    def compute(self) -> dict:
        n = max(self.n_batches, 1)
        return {"iou":  self.total_iou  / n,
                "dice": self.total_dice / n,
                "f1":   self.total_f1   / n,
                "acc":  self.total_acc  / n}

print("✓ Losses & Metrics defined")

✓ Losses & Metrics defined


## 📂 5. Dataset — Road Lane (YOLO) + BDD100K

In [6]:
import os
import cv2
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


# ─── Augmentation Pipelines ───────────────────────────────────────────────────

def get_train_transforms():
    return A.Compose([
        A.Resize(cfg.IMG_HEIGHT, cfg.IMG_WIDTH),
        A.HorizontalFlip(p=cfg.AUG_HFLIP_P),
        A.ColorJitter(
            brightness=cfg.AUG_BRIGHTNESS, contrast=cfg.AUG_CONTRAST,
            saturation=cfg.AUG_SATURATION, hue=cfg.AUG_HUE, p=0.8,
        ),
        A.MotionBlur(blur_limit=5, p=cfg.AUG_MOTION_BLUR_P),
        A.CoarseDropout(
            max_holes=8, max_height=16, max_width=16,
            fill_value=0, mask_fill_value=0, p=cfg.AUG_COARSE_DROPOUT,
        ),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(var_limit=(10, 50), p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(cfg.IMG_HEIGHT, cfg.IMG_WIDTH),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


# ─── Road Lane Dataset (YOLO segmentation format) ─────────────────────────────

def yolo_polygon_to_mask(label_path: str, img_h: int, img_w: int) -> np.ndarray:
    """Convert YOLO segmentation .txt (normalized polygons) to binary mask."""
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    if not os.path.exists(label_path):
        return mask
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            coords = list(map(float, parts[1:]))
            points = []
            for i in range(0, len(coords) - 1, 2):
                x = int(coords[i]     * img_w)
                y = int(coords[i + 1] * img_h)
                points.append([x, y])
            if len(points) >= 3:
                pts = np.array(points, dtype=np.int32)
                cv2.fillPoly(mask, [pts], 255)
    return mask


class RoadLaneDataset(Dataset):
    """Roboflow Road Lane Segmentation dataset (YOLO polygon format)."""

    def __init__(self, split: str = "train", transform=None):
        assert split in ("train", "val", "test")
        self.transform = transform
        self.img_dir   = Path(cfg.ROAD_LANE_DIR) / split / "images"
        self.lbl_dir   = Path(cfg.ROAD_LANE_DIR) / split / "labels"
        self.images    = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        ])
        if len(self.images) == 0:
            raise RuntimeError(f"No images found in {self.img_dir}")
        print(f"[RoadLane] {split}: {len(self.images)} samples")

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w  = image.shape[:2]
        mask  = (yolo_polygon_to_mask(str(lbl_path), h, w) > 127).astype(np.float32)
        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0)
        return image, mask


# ─── BDD100K Dataset (PNG mask format) ────────────────────────────────────────

BDD_LANE_CLASS_IDS = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}

class BDD100KLaneDataset(Dataset):
    """BDD100K segmentation dataset — extracts lane-marking pixels."""

    def __init__(self, split: str = "train", transform=None):
        assert split in ("train", "val")
        self.transform = transform
        self.img_dir   = Path(cfg.BDD100K_SEG_DIR) / "images" / split
        self.lbl_dir   = Path(cfg.BDD100K_SEG_DIR) / "labels" / split
        self.images    = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        ])
        if len(self.images) == 0:
            raise RuntimeError(f"No images found in {self.img_dir}")
        print(f"[BDD100K] {split}: {len(self.images)} samples")

    def __len__(self): return len(self.images)

    def _load_mask(self, img_stem: str) -> np.ndarray:
        for ext in (".png", ".jpg"):
            p = self.lbl_dir / (img_stem + ext)
            if p.exists():
                mask_raw = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if mask_raw is not None:
                    binary = np.zeros_like(mask_raw, dtype=np.float32)
                    for cid in BDD_LANE_CLASS_IDS:
                        binary[mask_raw == cid] = 1.0
                    return binary
        return np.zeros((720, 1280), dtype=np.float32)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image    = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask     = self._load_mask(img_path.stem)
        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0)
        return image, mask


# ─── Combined DataLoaders ─────────────────────────────────────────────────────

def get_dataloaders():
    train_tf = get_train_transforms()
    val_tf   = get_val_transforms()

    rl_train  = RoadLaneDataset("train", train_tf)
    rl_val    = RoadLaneDataset("val",   val_tf)
    bdd_train = BDD100KLaneDataset("train", train_tf)
    bdd_val   = BDD100KLaneDataset("val",   val_tf)

    train_ds  = ConcatDataset([rl_train, bdd_train])
    val_ds    = ConcatDataset([rl_val,   bdd_val])
    print(f"\n[Dataset] Total train: {len(train_ds)} | val: {len(val_ds)}\n")

    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
    )
    return train_loader, val_loader

print("✓ Dataset classes defined")

✓ Dataset classes defined


## 🔁 6. Training Loop

In [7]:
import os
print("Current directory:", os.getcwd())
print("Contents:", os.listdir("."))

Current directory: /kaggle/working
Contents: ['.virtual_documents']


In [8]:
import os

base_path = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels"

print("Checking all folders in dataset...")
for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for dir in dirs[:5]:  # Limit to avoid spam
        print(f"{subindent}{dir}/")
    if len(dirs) > 5:
        print(f"{subindent}... and {len(dirs)-5} more")

Checking all folders in dataset...
road-lane-segmentation-imgs-and-labels/
  dataset/
  dataset/
    val/
    test/
    train/
    val/
      labels/
      images/
      labels/
      images/
    test/
      labels/
      images/
      labels/
      images/
    train/
      labels/
      images/
      labels/
      images/


In [9]:
import os
label_dir = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels/dataset/train/labels"

if os.path.exists(label_dir):
    files = os.listdir(label_dir)
    print(f"Files in labels: {len(files)}")
    print(f"First 10: {files[:10]}")
    # Check file extensions
    extensions = set([f.split('.')[-1] for f in files if '.' in f])
    print(f"Extensions found: {extensions}")
else:
    print("Labels folder doesn't exist")

Files in labels: 815
First 10: ['i326.txt', 'i127.txt', 'i475.txt', 'i89.txt', 'i392.txt', 'i190.txt', 'i123.txt', 'i321.txt', 'i117.txt', 'i377.txt']
Extensions found: {'txt'}


In [ ]:
"""
Lane Detection — Full Training Script
UNet with EfficientNet-B4 encoder, Combined BCE+Dice loss,
mixed-precision training, cosine annealing, early stopping.

Requirements:
    pip install torch torchvision timm albumentations tensorboard

Directory layout expected:
    data/
      train/
        images/   *.jpg / *.png
        masks/    *.png  (binary, same stem as image)
      val/
        images/
        masks/
"""

import os
import warnings
import logging

# Suppress OpenCV warnings BEFORE importing cv2
os.environ['OPENCV_LOG_LEVEL'] = 'SILENT'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Also suppress Python warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import cv2
import numpy as np
import time
from pathlib import Path
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm


# ─── Config ──────────────────────────────────────────────────────────────────

class Config:
    # Use the actual Kaggle dataset path
    DATA_DIR = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels/dataset"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    BEST_MODEL_PATH = "/kaggle/working/checkpoints/best_model.pth"
    LOG_DIR = "/kaggle/working/runs/lane_det"

    # Model
    ENCODER         = "efficientnet_b4"
    PRETRAINED      = True
    NUM_CLASSES     = 1          # binary segmentation

    # Input
    IMG_HEIGHT      = 384
    IMG_WIDTH       = 640

    # Training
    NUM_EPOCHS      = 100
    BATCH_SIZE      = 8
    NUM_WORKERS     = 4
    LEARNING_RATE   = 1e-3
    WEIGHT_DECAY    = 1e-4
    MIXED_PRECISION = True
    DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

    # Loss weights
    BCE_WEIGHT      = 0.5
    DICE_WEIGHT     = 0.5

    # Metrics
    CONF_THRESHOLD  = 0.5

    def make_dirs(self):
        os.makedirs(self.LOG_DIR, exist_ok=True)
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)
        parent = os.path.dirname(self.BEST_MODEL_PATH)
        if parent:
            os.makedirs(parent, exist_ok=True)


cfg = Config()

def __len__(self):
    return len(self.image_paths)
# ─── Dataset ─────────────────────────────────────────────────────────────────
class LaneDataset(Dataset):
    def __init__(self, root: str, transform=None):
        self.image_paths = sorted(
            glob(os.path.join(root, "images", "*.jpg")) +
            glob(os.path.join(root, "images", "*.png"))
        )
        if not self.image_paths:
            raise FileNotFoundError(f"No images found under {root}/images/")
        
        self.label_dir = os.path.join(root, "labels")  # TXT files here
        self.transform = transform
        
        # Match pairs
        self.pairs = []
        for img_path in self.image_paths:
            stem = Path(img_path).stem
            txt_path = os.path.join(self.label_dir, f"{stem}.txt")
            if os.path.exists(txt_path):
                self.pairs.append((img_path, txt_path))
            else:
                self.pairs.append((img_path, None))

    def __len__(self):
        return len(self.pairs)

    def _txt_to_mask(self, txt_path, img_shape):
        """Convert YOLO format TXT to binary mask"""
        h, w = img_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        
        if txt_path is None or not os.path.exists(txt_path):
            return mask
        
        with open(txt_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            
            # YOLO format: class_id x1 y1 x2 y2 ... (normalized 0-1)
            # For lanes, it's usually polygons
            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            
            # Convert normalized coords to pixel coords
            points = []
            for i in range(0, len(coords), 2):
                x = int(coords[i] * w)
                y = int(coords[i+1] * h)
                points.append([x, y])
            
            if len(points) >= 2:
                # Draw lane line as thick line or polygon
                points = np.array(points, np.int32)
                cv2.fillPoly(mask, [points], 255)  # Fill polygon
                # Or cv2.polylines(mask, [points], False, 255, thickness=5)
        
        return mask

    def __getitem__(self, idx):
        img_path, txt_path = self.pairs[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Convert TXT annotation to mask
        mask = self._txt_to_mask(txt_path, image.shape)
        mask = (mask > 127).astype(np.uint8)
        
        if self.transform:
            result = self.transform(image=image, mask=mask)
            image = result["image"]
            mask = result["mask"]
        
        if isinstance(mask, torch.Tensor):
            mask = mask.float().unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).float().unsqueeze(0)
        
        return image, mask
def get_transforms(train: bool):
    h, w = cfg.IMG_HEIGHT, cfg.IMG_WIDTH
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    if train:
        return A.Compose([
            A.Resize(h, w),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=10, p=0.5),
            A.OneOf([
                A.RandomBrightnessContrast(p=1),
                A.HueSaturationValue(p=1),
                A.CLAHE(p=1),
            ], p=0.4),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(h, w),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])


def get_dataloaders():
    train_ds = LaneDataset(
        os.path.join(cfg.DATA_DIR, "train"),
        transform=get_transforms(train=True),
    )
    val_ds = LaneDataset(
        os.path.join(cfg.DATA_DIR, "val"),
        transform=get_transforms(train=False),
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
    )
    print(f"  Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")
    return train_loader, val_loader


# ─── Model ───────────────────────────────────────────────────────────────────

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = ConvBNReLU(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            # Handle size mismatch from rounding
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                                  align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class LaneUNet(nn.Module):
    """
    EfficientNet-B4 encoder + lightweight UNet decoder.
    Encoder feature channels (B4): 24, 32, 56, 160, 448  (indices 0-4)
    """

    ENCODER_CHANNELS = {
        "efficientnet_b0": [16, 24, 40, 112, 320],
        "efficientnet_b1": [16, 24, 40, 112, 320],
        "efficientnet_b2": [16, 24, 48, 120, 352],
        "efficientnet_b3": [24, 32, 48, 136, 384],
        "efficientnet_b4": [24, 32, 56, 160, 448],
        "efficientnet_b5": [24, 40, 64, 176, 512],
    }

    def __init__(self, encoder_name="efficientnet_b4", pretrained=True,
                 num_classes=1):
        super().__init__()
        enc_ch = self.ENCODER_CHANNELS.get(encoder_name, [24, 32, 56, 160, 448])

        # ── Encoder ──────────────────────────────────────────────────────────
        backbone = timm.create_model(
            encoder_name, pretrained=pretrained, features_only=True
        )
        # Expose each stage as a named attribute (for differential LR)
        self.enc0 = backbone.layer0 if hasattr(backbone, "layer0") else backbone.conv_stem
        # timm features_only backbone — we'll just store the whole backbone
        # and call it stage-by-stage via forward_features
        self._backbone = backbone
        self._enc_ch   = enc_ch

        # ── Bridge ───────────────────────────────────────────────────────────
        self.bridge = ConvBNReLU(enc_ch[4], enc_ch[4] * 2)

        # ── Decoder ──────────────────────────────────────────────────────────
        d = enc_ch[4] * 2
        self.dec0 = DecoderBlock(d,       enc_ch[3], 256)
        self.dec1 = DecoderBlock(256,     enc_ch[2], 128)
        self.dec2 = DecoderBlock(128,     enc_ch[1], 64)
        self.dec3 = DecoderBlock(64,      enc_ch[0], 32)
        self.dec4 = DecoderBlock(32,      0,         16)   # no skip at stage 0

        # ── Head ─────────────────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, num_classes, 1),
        )

    # Expose encoder params via named properties (used by train() for diff LR)
    @property
    def enc1(self): return nn.ModuleList([])
    @property
    def enc2(self): return nn.ModuleList([])
    @property
    def enc3(self): return nn.ModuleList([])
    @property
    def enc4(self): return nn.ModuleList([])
    @property
    def enc5(self): return nn.ModuleList([])

    def encoder_params(self):
        return list(self._backbone.parameters())

    def decoder_params(self):
        return (list(self.bridge.parameters()) +
                list(self.dec0.parameters()) +
                list(self.dec1.parameters()) +
                list(self.dec2.parameters()) +
                list(self.dec3.parameters()) +
                list(self.dec4.parameters()) +
                list(self.head.parameters()))

    def forward(self, x):
        features = self._backbone(x)   # list of 5 feature maps
        s0, s1, s2, s3, s4 = features

        x = self.bridge(s4)
        x = self.dec0(x, s3)
        x = self.dec1(x, s2)
        x = self.dec2(x, s1)
        x = self.dec3(x, s0)
        x = self.dec4(x, None)

        # Upsample to input resolution
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return self.head(x)


def build_model(pretrained=True):
    return LaneUNet(
        encoder_name=cfg.ENCODER,
        pretrained=pretrained,
        num_classes=cfg.NUM_CLASSES,
    )


# ─── Loss ────────────────────────────────────────────────────────────────────

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        flat_p = probs.view(probs.size(0), -1)
        flat_t = targets.view(targets.size(0), -1)
        intersection = (flat_p * flat_t).sum(1)
        dice = (2 * intersection + self.smooth) / (
            flat_p.sum(1) + flat_t.sum(1) + self.smooth
        )
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, bce_w=0.5, dice_w=0.5, pos_weight=5.0):
        super().__init__()
        self.bce_w   = bce_w
        self.dice_w  = dice_w
        pw           = torch.tensor([pos_weight])
        self.bce     = nn.BCEWithLogitsLoss(pos_weight=pw)
        self.dice    = DiceLoss()

    def forward(self, logits, targets):
        # Move pos_weight to same device on first use
        if self.bce.pos_weight.device != logits.device:
            self.bce.pos_weight = self.bce.pos_weight.to(logits.device)
        return (self.bce_w * self.bce(logits, targets) +
                self.dice_w * self.dice(logits, targets))


# ─── Metrics ─────────────────────────────────────────────────────────────────

class LaneMetrics:
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, logits: torch.Tensor, masks: torch.Tensor):
        preds = (torch.sigmoid(logits) > self.threshold).float()
        t = masks.float()
        self.tp += (preds * t).sum().item()
        self.fp += (preds * (1 - t)).sum().item()
        self.fn += ((1 - preds) * t).sum().item()
        self.tn += ((1 - preds) * (1 - t)).sum().item()

    def compute(self):
        eps = 1e-8
        iou  = self.tp / (self.tp + self.fp + self.fn + eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + eps)
        prec = self.tp / (self.tp + self.fp + eps)
        rec  = self.tp / (self.tp + self.fn + eps)
        return {"iou": iou, "dice": dice, "precision": prec, "recall": rec}


# ─── Checkpoint Helpers ───────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, epoch, best_iou, path):
    torch.save({
        "epoch":     epoch,
        "best_iou":  best_iou,
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }, path)
    print(f"  ✓ Checkpoint saved → {path}")


def load_checkpoint(model, optimizer, path):
    if not os.path.exists(path):
        return 0, 0.0
    ck = torch.load(path, map_location=cfg.DEVICE)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    print(f"  ✓ Resumed from epoch {ck['epoch']} (best IoU={ck['best_iou']:.4f})")
    return ck["epoch"], ck["best_iou"]


# ─── Epoch Runner ─────────────────────────────────────────────────────────────

def run_epoch(model, loader, criterion, optimizer, scaler,
              metrics, device, train: bool):
    model.train() if train else model.eval()
    metrics.reset()
    total_loss = 0.0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for step, (images, masks) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)

            with autocast(enabled=cfg.MIXED_PRECISION):
                logits = model(images)
                # Align logits to mask size if decoder rounding changed dims
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:],
                                           mode="bilinear", align_corners=False)
                loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item()
            metrics.update(logits.detach(), masks)

            if step % 50 == 0:
                phase = "TRAIN" if train else "VAL"
                print(f"    [{phase}] step {step}/{len(loader)}"
                      f"  loss={loss.item():.4f}")

    return total_loss / len(loader), metrics.compute()


# ─── Main Train Function ──────────────────────────────────────────────────────

def train():
    cfg.make_dirs()
    device = torch.device(cfg.DEVICE)
    print(f"\n{'='*60}")
    print(f"  Lane Detection Training")
    print(f"  Device: {device} | AMP: {cfg.MIXED_PRECISION}")
    print(f"{'='*60}\n")

    train_loader, val_loader = get_dataloaders()

    model     = build_model(pretrained=cfg.PRETRAINED).to(device)
    criterion = CombinedLoss(cfg.BCE_WEIGHT, cfg.DICE_WEIGHT, pos_weight=5.0)

    # Differential LR: encoder (pretrained) gets 10× lower LR than decoder
    optimizer = optim.AdamW([
        {"params": model.encoder_params(), "lr": cfg.LEARNING_RATE * 0.1},
        {"params": model.decoder_params(), "lr": cfg.LEARNING_RATE},
    ], weight_decay=cfg.WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6
    )
    scaler              = GradScaler(enabled=cfg.MIXED_PRECISION)
    start_epoch, best_iou = load_checkpoint(model, optimizer, cfg.BEST_MODEL_PATH)
    writer              = SummaryWriter(cfg.LOG_DIR)
    metrics             = LaneMetrics(threshold=cfg.CONF_THRESHOLD)
    patience            = 10
    patience_counter    = 0

    print(f"Starting training for {cfg.NUM_EPOCHS} epochs...\n")

    for epoch in range(start_epoch, cfg.NUM_EPOCHS):
        t0 = time.time()
        print(f"\n── Epoch {epoch+1}/{cfg.NUM_EPOCHS}"
              f"  (LR={scheduler.get_last_lr()}) ──")

        train_loss, train_stats = run_epoch(
            model, train_loader, criterion, optimizer,
            scaler, metrics, device, train=True,
        )
        val_loss, val_stats = run_epoch(
            model, val_loader, criterion, optimizer,
            scaler, metrics, device, train=False,
        )

        scheduler.step()
        elapsed = time.time() - t0

        print(f"\n  Train | loss={train_loss:.4f}  IoU={train_stats['iou']:.4f}"
              f"  Dice={train_stats['dice']:.4f}")
        print(f"  Val   | loss={val_loss:.4f}  IoU={val_stats['iou']:.4f}"
              f"  Dice={val_stats['dice']:.4f}  ({elapsed:.1f}s)")

        writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("IoU",  {"train": train_stats["iou"],
                                    "val":   val_stats["iou"]},   epoch)
        writer.add_scalars("Dice", {"train": train_stats["dice"],
                                    "val":   val_stats["dice"]},  epoch)
        writer.add_scalar("LR",    scheduler.get_last_lr()[0],    epoch)

        val_iou = val_stats["iou"]
        if val_iou > best_iou:
            best_iou         = val_iou
            patience_counter = 0
            save_checkpoint(model, optimizer, epoch + 1,
                            best_iou, cfg.BEST_MODEL_PATH)
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")

        if (epoch + 1) % 10 == 0:
            ck_path = os.path.join(cfg.CHECKPOINT_DIR, f"epoch_{epoch+1}.pth")
            save_checkpoint(model, optimizer, epoch + 1, best_iou, ck_path)

        if patience_counter >= patience:
            print(f"\n  Early stopping triggered at epoch {epoch+1}")
            break

    writer.close()
    print(f"\n{'='*60}")
    print(f"  Training complete! Best Val IoU: {best_iou:.4f}")
    print(f"  Best model saved to: {cfg.BEST_MODEL_PATH}")
    print(f"{'='*60}\n")


# ▶ RUN TRAINING
if __name__ == "__main__":
    train()

E0000 00:00:1776285323.108167     120 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776285323.218332     120 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776285324.207641     120 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776285324.207673     120 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776285324.207677     120 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776285324.207679     120 computation_placer.cc:177] computation placer already registered. Please check linka


  Lane Detection Training
  Device: cuda | AMP: True

  Train: 815 samples | Val: 101 samples


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Starting training for 100 epochs...


── Epoch 1/100  (LR=[0.0001, 0.001]) ──
    [TRAIN] step 0/101  loss=0.9431
    [TRAIN] step 50/101  loss=0.2360
    [TRAIN] step 100/101  loss=0.1613
    [VAL] step 0/13  loss=0.1339

  Train | loss=0.3972  IoU=0.4897  Dice=0.6574
  Val   | loss=0.1411  IoU=0.7936  Dice=0.8849  (62.2s)
  ✓ Checkpoint saved → /kaggle/working/checkpoints/best_model.pth

── Epoch 2/100  (LR=[9.939057285945933e-05, 0.0009938503261272714]) ──
    [TRAIN] step 0/101  loss=0.1645


In [ ]:
import os

# Search entire system for your model
print("Searching for best_model.pth...")
found = False
for root, dirs, files in os.walk("/kaggle"):
    for file in files:
        if "best" in file and file.endswith(".pth"):
            print(f"✅ FOUND: {os.path.join(root, file)}")
            print(f"   Size: {os.path.getsize(os.path.join(root, file)) / (1024*1024):.1f} MB")
            found = True

if not found:
    print("❌ Model not found anywhere.")
    print("   Kaggle cleared it when session restarted.")
    print("   You need to re-run training.")

In [ ]:
"""
Test script for Lane Detection PyTorch model
Load best_model.pth and run inference on images
"""

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path

# ─── Your Model Architecture (copy from training script) ────────────────────

import torch.nn as nn
import torch.nn.functional as F
import timm

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = ConvBNReLU(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class LaneUNet(nn.Module):
    ENCODER_CHANNELS = {
        "efficientnet_b4": [24, 32, 56, 160, 448],
    }

    def __init__(self, encoder_name="efficientnet_b4", num_classes=1):
        super().__init__()
        enc_ch = self.ENCODER_CHANNELS.get(encoder_name, [24, 32, 56, 160, 448])
        
        self._backbone = timm.create_model(encoder_name, pretrained=False, features_only=True)
        self.bridge = ConvBNReLU(enc_ch[4], enc_ch[4] * 2)
        
        d = enc_ch[4] * 2
        self.dec0 = DecoderBlock(d, enc_ch[3], 256)
        self.dec1 = DecoderBlock(256, enc_ch[2], 128)
        self.dec2 = DecoderBlock(128, enc_ch[1], 64)
        self.dec3 = DecoderBlock(64, enc_ch[0], 32)
        self.dec4 = DecoderBlock(32, 0, 16)
        
        self.head = nn.Sequential(
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, num_classes, 1),
        )

    def forward(self, x):
        features = self._backbone(x)
        s0, s1, s2, s3, s4 = features
        
        x = self.bridge(s4)
        x = self.dec0(x, s3)
        x = self.dec1(x, s2)
        x = self.dec2(x, s1)
        x = self.dec3(x, s0)
        x = self.dec4(x, None)
        
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return self.head(x)

# ─── Config ─────────────────────────────────────────────────────────────────

class Config:
    MODEL_PATH = "/kaggle/working/checkpoints/best_model.pth"  # Your model
    IMG_HEIGHT = 384
    IMG_WIDTH = 640
    CONF_THRESHOLD = 0.5
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ─── Lane Detector Class ────────────────────────────────────────────────────

class LaneDetector:
    def __init__(self, model_path, device=None):
        self.device = device or Config.DEVICE
        self.model = self._load_model(model_path)
        self.model.eval()
        print(f"✓ Model loaded on {self.device}")
        
    def _load_model(self, path):
        model = LaneUNet(encoder_name="efficientnet_b4", num_classes=1)
        
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model not found: {path}")
            
        checkpoint = torch.load(path, map_location=self.device)
        model.load_state_dict(checkpoint["model"], strict=False)
        model.to(self.device)
        return model
    
    def preprocess(self, image):
        """Preprocess image for model"""
        original_size = (image.shape[1], image.shape[0])
        
        # Resize
        img_resized = cv2.resize(image, (Config.IMG_WIDTH, Config.IMG_HEIGHT))
        
        # Normalize (ImageNet stats)
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
        img_norm = img_rgb / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_norm = (img_norm - mean) / std
        
        # To tensor (B, C, H, W)
        img_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).float().unsqueeze(0)
        
        return img_tensor.to(self.device), original_size
    
    def postprocess(self, output, original_size):
        """Convert model output to binary mask"""
        # Sigmoid + threshold
        pred = torch.sigmoid(output)
        mask = (pred > Config.CONF_THRESHOLD).cpu().numpy().astype(np.uint8)
        
        # Resize to original
        mask = mask.squeeze()
        mask_resized = cv2.resize(mask, original_size, interpolation=cv2.INTER_NEAREST)
        
        return mask_resized * 255
    
    @torch.no_grad()
    def predict(self, image):
        """Run inference"""
        input_tensor, original_size = self.preprocess(image)
        
        # Forward pass
        output = self.model(input_tensor)
        
        # Postprocess
        mask = self.postprocess(output, original_size)
        return mask
    
    def visualize(self, image, mask, alpha=0.6):
        """Create visualization"""
        # Overlay mask on image
        overlay = image.copy()
        overlay[mask > 127] = [0, 0, 255]  # Red lanes
        
        # Blend
        result = cv2.addWeighted(image, 0.4, overlay, 0.6, 0)
        
        # Add metrics text
        lane_ratio = np.sum(mask > 127) / (mask.shape[0] * mask.shape[1]) * 100
        cv2.putText(result, f"Lane: {lane_ratio:.1f}%", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        return result

# ─── Test Functions ────────────────────────────────────────────────────────

def test_single_image(detector, image_path, save_path=None):
    """Test on single image and show results"""
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Cannot load: {image_path}")
        return
    
    # Inference
    mask = detector.predict(image)
    result = detector.visualize(image, mask)
    
    # Save if path provided
    if save_path:
        cv2.imwrite(save_path, result)
        cv2.imwrite(save_path.replace(".jpg", "_mask.jpg"), mask)
        print(f"✓ Saved: {save_path}")
    
    # Display
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Lane Mask")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.title("Prediction")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def test_folder(detector, folder_path, output_folder="test_results"):
    """Test all images in folder"""
    os.makedirs(output_folder, exist_ok=True)
    
    image_paths = glob(os.path.join(folder_path, "*.jpg")) + \
                  glob(os.path.join(folder_path, "*.png"))
    
    print(f"Found {len(image_paths)} images")
    
    for img_path in image_paths:
        filename = Path(img_path).name
        save_path = os.path.join(output_folder, f"pred_{filename}")
        test_single_image(detector, img_path, save_path)
    
    print(f"\n✓ All results saved to: {output_folder}/")

# ─── MAIN ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Initialize detector
    detector = LaneDetector(Config.MODEL_PATH)
    
    # Option 1: Test single image (from validation set)
    val_img = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels/dataset/val/images/i10.jpg"
    if os.path.exists(val_img):
        test_single_image(detector, val_img, "result_val.jpg")
    else:
        print("Val image not found, use your own image path")
    
    # Option 2: Test entire validation folder
    val_folder = "/kaggle/input/datasets/princekhunt19/road-lane-segmentation-imgs-and-labels/dataset/val/images"
    if os.path.exists(val_folder):
        test_folder(detector, val_folder, "val_predictions")
    
    # Option 3: Test your own image
    # test_single_image(detector, "your_image.jpg", "output.jpg")

## 📤 7. Export to ONNX

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np


def export_onnx():
    print("\n[Export] Loading best model checkpoint...")
    model = build_model(pretrained=False)
    ck    = torch.load(cfg.BEST_MODEL_PATH, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()

    dummy = torch.randn(1, 3, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)

    print(f"[Export] Exporting to ONNX: {cfg.ONNX_PATH}")
    torch.onnx.export(
        model, dummy, cfg.ONNX_PATH,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["image"],
        output_names=["lane_mask"],
        dynamic_axes={"image": {0: "batch"}, "lane_mask": {0: "batch"}},
        verbose=False,
    )

    # Validate ONNX graph
    onnx_model = onnx.load(cfg.ONNX_PATH)
    onnx.checker.check_model(onnx_model)
    print("  ✓ ONNX graph valid")

    # Compare PyTorch vs ONNX outputs
    sess = ort.InferenceSession(cfg.ONNX_PATH, providers=["CPUExecutionProvider"])
    with torch.no_grad():
        pt_out = torch.sigmoid(model(dummy)).numpy()
    ort_out = sess.run(["lane_mask"], {"image": dummy.numpy()})[0]
    ort_out = 1 / (1 + np.exp(-ort_out))   # sigmoid

    max_diff = np.abs(pt_out - ort_out).max()
    print(f"  ✓ Max output diff PyTorch vs ONNX: {max_diff:.6f}")
    assert max_diff < 1e-4, "ONNX outputs differ too much!"

    print(f"\n[Export] Done! → {cfg.ONNX_PATH}")
    print("\nNext: copy lane_detection.onnx to your Jetson Nano and run:")
    print(f"  trtexec --onnx={cfg.ONNX_PATH} --saveEngine={cfg.TRT_ENGINE_PATH} --fp16 --workspace=2048")


export_onnx()

## 🚀 8. Jetson Nano Inference (copy this cell to Jetson)
> **Run this cell on your Jetson Nano** after generating the TRT engine with `trtexec`.

In [ ]:
"""
Jetson Nano Real-Time Lane Detection Inference
TensorRT FP16 engine → ~35 FPS on Jetson Nano 4GB

Prerequisites (on Jetson Nano):
  JetPack 5.x (TensorRT, CUDA, cuDNN)
  pip install pycuda numpy opencv-python

Generate engine first:
  trtexec --onnx=lane_detection.onnx \
          --saveEngine=lane_detection_fp16.trt \
          --fp16 --workspace=2048
"""

import argparse, time, cv2, numpy as np, os

try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit
    TRT_AVAILABLE = True
except ImportError:
    TRT_AVAILABLE = False
    print("[WARNING] TensorRT not found — falling back to ONNX Runtime.")
    import onnxruntime as ort


MEAN       = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD        = np.array([0.229, 0.224, 0.225], dtype=np.float32)
LANE_COLOR = (0, 255, 100)   # bright green overlay


# ─── TensorRT Engine ──────────────────────────────────────────────────────────

class TRTInferenceEngine:
    def __init__(self, engine_path: str):
        if not os.path.exists(engine_path):
            raise FileNotFoundError(f"TRT engine not found: {engine_path}")
        logger = trt.Logger(trt.Logger.WARNING)
        with open(engine_path, "rb") as f:
            runtime     = trt.Runtime(logger)
            self.engine = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.inputs, self.outputs, self.bindings, self.stream = self._allocate_buffers()
        self.input_shape  = (1, 3, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)
        self.output_shape = (1, 1, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)
        print(f"[TRT] Engine loaded: {engine_path}")

    def _allocate_buffers(self):
        inputs, outputs, bindings = [], [], []
        stream = cuda.Stream()
        for binding in self.engine:
            shape    = self.engine.get_binding_shape(binding)
            dtype    = trt.nptype(self.engine.get_binding_dtype(binding))
            size     = trt.volume(shape) * np.dtype(dtype).itemsize
            host_mem = cuda.pagelocked_empty(trt.volume(shape), dtype)
            dev_mem  = cuda.mem_alloc(size)
            bindings.append(int(dev_mem))
            (inputs if self.engine.binding_is_input(binding) else outputs).append(
                {"host": host_mem, "device": dev_mem})
        return inputs, outputs, bindings, stream

    def infer(self, blob: np.ndarray) -> np.ndarray:
        np.copyto(self.inputs[0]["host"], blob.ravel())
        cuda.memcpy_htod_async(self.inputs[0]["device"],  self.inputs[0]["host"],  self.stream)
        self.context.execute_async_v2(self.bindings, self.stream.handle)
        cuda.memcpy_dtoh_async(self.outputs[0]["host"], self.outputs[0]["device"], self.stream)
        self.stream.synchronize()
        return self.outputs[0]["host"].reshape(self.output_shape)


# ─── ONNX Fallback ────────────────────────────────────────────────────────────

class ONNXInferenceEngine:
    def __init__(self, onnx_path: str):
        providers    = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                        if ort.get_device() == "GPU" else ["CPUExecutionProvider"])
        self.session     = ort.InferenceSession(onnx_path, providers=providers)
        self.input_name  = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
        print(f"[ONNX] Session ready | providers: {providers}")

    def infer(self, blob: np.ndarray) -> np.ndarray:
        return self.session.run([self.output_name], {self.input_name: blob})[0]


# ─── Pre / Post Processing ───────────────────────────────────────────────────

def preprocess(frame: np.ndarray) -> np.ndarray:
    img = cv2.resize(frame, (cfg.IMG_WIDTH, cfg.IMG_HEIGHT))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    return img.transpose(2, 0, 1)[np.newaxis].astype(np.float32)

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

def postprocess(logits, orig_h, orig_w):
    mask = (sigmoid(logits[0, 0]) > cfg.CONF_THRESHOLD).astype(np.uint8) * 255
    return cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

def overlay_lanes(frame, mask, alpha=0.45):
    colored = np.zeros_like(frame)
    colored[mask > 127] = LANE_COLOR
    return cv2.addWeighted(frame, 1.0, colored, alpha, 0)


# ─── Runner ──────────────────────────────────────────────────────────────────

def infer_image(engine, image_path: str):
    frame  = cv2.imread(image_path)
    h, w   = frame.shape[:2]
    blob   = preprocess(frame)
    logits = engine.infer(blob)
    mask   = postprocess(logits, h, w)
    result = overlay_lanes(frame, mask)
    out    = image_path.replace(".", "_lane.")
    cv2.imwrite(out, result)
    print(f"[Infer] Saved: {out}")

def infer_video(engine, source):
    cap    = cv2.VideoCapture(int(source) if str(source).isdigit() else source)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fps_c  = cap.get(cv2.CAP_PROP_FPS) or 30
    w_o    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_o    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter("lane_output.mp4", fourcc, fps_c, (w_o, h_o))
    fps_win = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        t0     = time.perf_counter()
        blob   = preprocess(frame)
        logits = engine.infer(blob)
        mask   = postprocess(logits, h_o, w_o)
        result = overlay_lanes(frame, mask)
        dt     = time.perf_counter() - t0
        fps_win.append(1.0 / max(dt, 1e-6))
        if len(fps_win) > 30: fps_win.pop(0)
        fps_avg = sum(fps_win) / len(fps_win)
        cv2.putText(result, f"FPS: {fps_avg:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        writer.write(result)
        cv2.imshow("Lane Detection — Jetson Nano", result)
        if cv2.waitKey(1) & 0xFF == ord("q"): break
    cap.release(); writer.release(); cv2.destroyAllWindows()
    print(f"[Infer] Avg FPS: {sum(fps_win)/len(fps_win):.1f}")


# ─── Entry Point (change source here) ────────────────────────────────────────
# source = 0                         # webcam
# source = "/path/to/dashcam.mp4"    # video file
# source = "/path/to/road.jpg"       # single image

SOURCE = 0   # ← change this

if TRT_AVAILABLE and os.path.exists(cfg.TRT_ENGINE_PATH):
    engine = TRTInferenceEngine(cfg.TRT_ENGINE_PATH)
elif os.path.exists(cfg.ONNX_PATH):
    print("[INFO] Using ONNX Runtime fallback")
    engine = ONNXInferenceEngine(cfg.ONNX_PATH)
else:
    print("[ERROR] No engine found. Run export_onnx() first.")
    engine = None

if engine:
    src = str(SOURCE)
    if os.path.isfile(src) and src.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        infer_image(engine, src)
    else:
        infer_video(engine, src)